In [1]:
import pandas as pd
import sys, os
from pathlib import Path

import os
import math
sys.path.append(str(Path(os.getcwd()).parent))
from utils.wrappers import measure_time_and_space, measure_time
from typing import Iterable, List
from utils.data_structures import RollingMeanArray, UpwardsDownwardsArray

### Getting list of filepaths

In [2]:
@measure_time_and_space
def select_target_csvs(directory):
    # Define target filenames (use a set for O(1) lookup)
    target_files = {
        "MSFT_data.csv",
        "NVDA_data.csv",
        "AAPL_data.csv",
        "GOOGL_data.csv",
        "AMZN_data.csv",
        "META_data.csv",
        "TSLA_data.csv",
    }

    selected = []
    with os.scandir(directory) as entries:
        for entry in entries:
            if entry.is_file() and entry.name in target_files:
                selected.append(entry.path)
    return selected

@measure_time_and_space
def select_target_csvs_nonrecursive(directory):
    # Define target filenames (use a set for O(1) lookup)
    target_files = [
        "MSFT_data.csv",
        "NVDA_data.csv",
        "AAPL_data.csv",
        "GOOGL_data.csv",
        "AMZN_data.csv",
        "META_data.csv",
        "TSLA_data.csv",
        ]
    

    selected_paths = []

    # Efficiently iterate through directory (not recursive)
    for filename in os.listdir(directory):
        if filename in target_files:
            selected_paths.append(os.path.join(directory, filename))

    return selected_paths

In [3]:
select_target_csvs(Path.cwd() / "csv")
select_target_csvs_nonrecursive(Path.cwd() / "csv")



[select_target_csvs] Time elapsed: 0.001378 seconds
[select_target_csvs] Peak memory: 3.33 KB
[select_target_csvs_nonrecursive] Time elapsed: 0.000960 seconds
[select_target_csvs_nonrecursive] Peak memory: 35.92 KB


['c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\AAPL_data.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\AMZN_data.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\GOOGL_data.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\MSFT_data.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\NVDA_data.csv']

In [4]:
file_paths = select_target_csvs(Path.cwd() / "csv")

[select_target_csvs] Time elapsed: 0.001754 seconds
[select_target_csvs] Peak memory: 3.21 KB


### Reading the selected csv files in a efficient manner

In [5]:
@measure_time_and_space

def read_filtered_csvs(file_paths, chunksize=100000):
    """
    Reads multiple CSV files in chunks, filters rows with date.year > 2022,
    and returns a single combined DataFrame.
    """
    combined_chunks = []  # list to store filtered chunks

    for path in file_paths:
        for chunk in pd.read_csv(path, parse_dates=['date'], chunksize=chunksize):
            filtered_chunk = chunk[chunk['date'].dt.year > 2022] # Filter rows where year > 2024
            if not filtered_chunk.empty:
                combined_chunks.append(filtered_chunk)

    # Concatenate all filtered chunks into a single DataFrame
    combined_df = pd.concat(combined_chunks, ignore_index=True) if combined_chunks else pd.DataFrame()
    return combined_df

combined_df = read_filtered_csvs(file_paths)

[read_filtered_csvs] Time elapsed: 0.034446 seconds
[read_filtered_csvs] Peak memory: 721.75 KB


In [6]:
combined_df.head()

,date,open,high,low,close,volume,Name
0,2023-01-04,102.61,105.368,102.00,105.35,67649387,AAPL
1,2023-01-05,105.75,105.850,102.41,102.71,55790992,AAPL
2,2023-01-06,100.56,102.370,99.87,100.70,68457388,AAPL
3,2023-01-07,98.68,100.130,96.43,96.45,81094428,AAPL
4,2023-01-08,98.55,99.110,96.76,96.96,70798016,AAPL


In [7]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2645 entries, 0 to 2644
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    2645 non-null   datetime64[ns]
 1   open    2645 non-null   float64       
 2   high    2645 non-null   float64       
 3   low     2645 non-null   float64       
 4   close   2645 non-null   float64       
 5   volume  2645 non-null   int64         
 6   Name    2645 non-null   object        
dtypes: datetime64[ns](1), float64(4), int64(1), object(1)
memory usage: 144.8+ KB


In [8]:
# Convert date datatype from object to datetime
combined_df = combined_df[combined_df["date"] < "2025-09-01"]
print("Data ranges from", combined_df["date"].min(), "to", combined_df["date"].max())


Data ranges from 2023-01-04 00:00:00 to 2025-02-07 00:00:00


In [9]:
@measure_time_and_space
def sma_pandas(df, window=20):
    """
    Calculate Simple Moving Average (SMA) for the 'close' column.
    """
    return df['close'].rolling(window=window).mean()

combined_df["SMA_30"] = sma_pandas(combined_df, window=30)

[sma_pandas] Time elapsed: 0.000514 seconds
[sma_pandas] Peak memory: 66.63 KB


In [10]:
rma = RollingMeanArray(combined_df['close'],30)

In [11]:
@measure_time_and_space
def sma_rma():
    """
    Calculate Simple Moving Average (SMA) for the 'close' column using RollingMeanArray class.
    """
    return pd.Series(rma.rolling_mean())
combined_df["SMA_30"] = sma_rma()

[sma_rma] Time elapsed: 0.003708 seconds
[sma_rma] Peak memory: 233.58 KB


In [12]:
@measure_time_and_space
def sma_naive():
    """
    Calculate Simple Moving Average (SMA) for the 'close' column using RollingMeanArray.naive_rolling_mean method, as a benchmark against the sma_rma method.
    """
    return pd.Series(rma.naive_rolling_mean())
combined_df["SMA_30"] = sma_naive()

[sma_naive] Time elapsed: 0.006855 seconds
[sma_naive] Peak memory: 231.37 KB


In [13]:
rma.window = 90
combined_df["SMA_90"] = sma_rma()
rma.window = 180
combined_df["SMA_180"] = sma_rma()

[sma_rma] Time elapsed: 0.003649 seconds
[sma_rma] Peak memory: 229.98 KB
[sma_rma] Time elapsed: 0.003638 seconds
[sma_rma] Peak memory: 227.84 KB


In [14]:
uda = UpwardsDownwardsArray(combined_df['close'])

In [15]:
@measure_time_and_space
def create_run_group():
    """
    Calculate the difference between consecutive 'close' prices in-place.
    """
    return uda.create_run_group()

@measure_time_and_space
def create_run_group_naive():
    """
    Calculate the difference between consecutive 'close' prices in-place.
    """
    return uda.create_run_group_naive()
create_run_group()


[create_run_group] Time elapsed: 0.002079 seconds
[create_run_group] Peak memory: 0.16 KB


[nan,
 -1,
 -1,
 -1,
 1,
 1,
 1,
 -1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 -1,
 -1,
 1,
 1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 1,
 1,
 -1,
 1,
 1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 1,
 1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 -1,
 1,
 1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 1,
 1,
 1,
 1,
 -1,
 -1,
 -1,
 1,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 1,
 -1,
 -1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 1,
 1,
 1,
 1,
 -1,
 -1,
 -1,
 -1,
 1,
 1,
 1,
 -1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 1,
 1,
 1,
 1,
 -1,
 1,
 1,
 1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 1,
 1,
 -1,
 -1,
 -1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 1,
 1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 -1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 -1,
 -1,
 -1,
 1,
 1,
 1,
 -1,
 1,
 -1,
 -1,
 1,
 1,
 1,
 1,
 -1,
 -1,
 -1,
 -1,
 1,
 -1,
 1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 -1,
 1,
 -1,
 -1,
 -1,
 -1,
 -1,
 1,
 1,
 -1,
 -1,
 -1,
 -1,
 -1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 1,
 -1,

In [17]:
create_run_group_naive()

[create_run_group_naive] Time elapsed: 0.001788 seconds
[create_run_group_naive] Peak memory: 18.20 KB


[nan,
 0,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,


In [27]:
combined_df.to_csv("mag7_stocks.csv")